# CoreForge AI Evaluation v2 — Hindi knowledge-grounded comparison

**No training or production deployment.** This evaluation tests one unchanged model—**Qwen3-4B-Instruct-2507**—on **40 Hindi QA questions**, with and without retrieved knowledge passages. It examines whether relevant reference text improves reliability.

**Important limits:** The reference knowledge is an *authored candidate set awaiting project-owner signoff*, not an official style guide or independently audited external corpus. Retrieval uses character TF-IDF over the reference set (no oracle lookup using answer IDs). The expected reference IDs are used **only for evaluation**. The two generations use matching per-question random seeds, but sampling remains stochastic. Human review is required; automatic metrics alone cannot prove accuracy.

Requires Colab T4 GPU, free quota not guaranteed. Run cells in order. Files are kept in Colab session and downloaded by you; **no model weights or results are uploaded to Adatacore**.


In [ ]:
import torch, shutil
assert torch.cuda.is_available(), "Please select Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Available GPU GiB:",round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
print("Free disk GiB:",round(shutil.disk_usage('/content').free/2**30,1))


## 1 — Install packages and retrieve benchmark files

This notebook does not require Unsloth or any CoreForge training script. It loads a quantized model using Hugging Face Transformers directly. It downloads the two GitHub JSON files so the test uses versioned knowledge and questions.


In [ ]:
%pip -q install 'transformers>=4.57,<5' 'accelerate>=1.8,<2' 'bitsandbytes>=0.46' 'scikit-learn>=1.4,<2' 'safetensors>=0.5'


In [ ]:
import json, hashlib, urllib.request
from pathlib import Path
ROOT=Path('/content/coreforge-eval-v2')
ROOT.mkdir(exist_ok=True)
BASE_URL='https://raw.githubusercontent.com/ajay333X/my-adatacore-site/main/workers/'
for name in ('coreforge_eval_v2_knowledge.json','coreforge_eval_v2_questions.json'):
    dest=ROOT/name
    urllib.request.urlretrieve(BASE_URL+name,dest)
knowledge=json.loads((ROOT/'coreforge_eval_v2_knowledge.json').read_text(encoding='utf-8'))
bank=json.loads((ROOT/'coreforge_eval_v2_questions.json').read_text(encoding='utf-8'))
entries=knowledge['entries']; questions=bank['questions']
assert len(questions)==40 and len(entries)>=12
ids={x['id'] for x in entries}
assert all(set(q['reference_ids']).issubset(ids) for q in questions)
print('Knowledge entries:',len(entries),'Benchmark questions:',len(questions))
print('Reference status:',knowledge['status'])
print('Question bank SHA256:',hashlib.sha256((ROOT/'coreforge_eval_v2_questions.json').read_bytes()).hexdigest())


## 2 — Load the untouched Qwen3-4B Instruct model

4-bit quantization is used to fit common Colab T4 memory limits. If GPU memory is insufficient, do not proceed to training or buy hardware—share the error. We use the model's instructed chat template without a thinking-mode flag.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID='Qwen/Qwen3-4B-Instruct-2507'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,quantization_config=quant,device_map='auto',
    torch_dtype=torch.float16,trust_remote_code=False
)
model.eval()
eos=tokenizer.convert_tokens_to_ids('<|im_end|>')
if not isinstance(eos,int) or eos<0: raise ValueError('Missing Qwen chat-end token; stop and check tokenizer version.')
print('Loaded:',MODEL_ID,'stop token:',eos)


## 3 — Retrieve reference passages by question similarity

The same question is used in both conditions. For the grounded condition only, character TF-IDF finds the **two most similar** knowledge passages. This is actual automatic retrieval, not selecting references from the benchmark answer key. The key IDs are saved separately to evaluate retrieval coverage.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
vectorizer=TfidfVectorizer(analyzer='char',ngram_range=(2,4),min_df=1)
documents=[e['title']+'\n'+e['content'] for e in entries]
matrix=vectorizer.fit_transform(documents)
def retrieve(prompt,k=2):
    similarity=cosine_similarity(vectorizer.transform([prompt]),matrix).ravel()
    picks=sorted(range(len(entries)),key=lambda i:(-float(similarity[i]),i))[:k]
    return [{**entries[i],'similarity':round(float(similarity[i]),4)} for i in picks]
for q in questions[:4]:
    docs=retrieve(q['prompt'])
    print(q['id'],'expected:',q['reference_ids'],'retrieved:',[(d['id'],d['similarity']) for d in docs])


## 4 — Paired generation: 40 questions × two conditions

Set `QUESTION_LIMIT=5` to run a quick smoke test first; default `40` runs the full experiment. Each pair uses the same seed and instruction, but different provided context. Answers are limited to 160 tokens and instructed to use at most two short Hindi sentences. Results are saved after each pair, so restarting this cell in the same runtime skips completed questions.


In [ ]:
import re, time, collections
QUESTION_LIMIT=40
assert 1<=QUESTION_LIMIT<=40
OUT=ROOT/'coreforge-eval-v2-partial.json'
SYSTEM="आप एक हिंदी QA सहायक हैं। केवल सवाल का संक्षिप्त, तथ्यात्मक उत्तर दें (अधिकतम दो छोटे वाक्य)। किसी जानकारी की पुष्टि न हो तो साफ कहें; तथ्य न गढ़ें।"
CONTEXT_RULES="संदर्भ जानकारी केवल तथ्य-जाँच के लिए है। अगर संदर्भ अपूर्ण है तो न गढ़ें। संदर्भ के बाहर प्रमाणित जानकारी न हो तो अनिश्चितता बताएं।"
def make_chat(question,docs=None):
    sysmsg=SYSTEM
    if docs:
        excerpts='\n\n'.join(f"[{d['id']}] {d['title']}: {d['content']}" for d in docs)
        sysmsg+='\n'+CONTEXT_RULES+'\nसंदर्भ:\n'+excerpts
    return tokenizer.apply_chat_template(
        [{"role":"system","content":sysmsg},{"role":"user","content":question}],
        tokenize=False,add_generation_prompt=True)
def repetition_ratio(answer):
    tokens=re.findall(r"\S+",answer.lower())
    if len(tokens)<7:return 0.0
    grams=[tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
    return round(1.0-len(set(grams))/len(grams),3)
def generate_answer(question,docs,seed):
    torch.manual_seed(seed)
    prompt=make_chat(question,docs)
    encoded=tokenizer(prompt,return_tensors='pt')
    encoded={k:v.to(model.device) for k,v in encoded.items()}
    with torch.inference_mode():
        generated=model.generate(**encoded,max_new_tokens=160,
            do_sample=True,temperature=0.7,top_p=0.8,top_k=20,
            eos_token_id=eos,pad_token_id=eos)
    ids=generated[0][encoded['input_ids'].shape[-1]:].tolist()
    text=tokenizer.decode(ids,skip_special_tokens=True).strip()
    return {"answer":text,"tokens":len(ids),"stopped_on_eos":bool(ids and ids[-1]==eos),
        "hit_limit":bool(len(ids)>=160 and ids[-1]!=eos),
        "replacement_characters":text.count("\ufffd"),
        "repetition_ratio":repetition_ratio(text)}
if OUT.exists():
    old=json.loads(OUT.read_text(encoding='utf-8'))
    assert old.get('model')==MODEL_ID and old.get('version')==bank['version'], "Partial results belong to another experiment."
    results=old['results']
else:
    results=[]
done={r['id'] for r in results}
for i,q in enumerate(questions[:QUESTION_LIMIT]):
    if q['id'] in done: continue
    refs=retrieve(q['prompt'],k=2)
    base=generate_answer(q['prompt'],None,20261113+i)
    grounded=generate_answer(q['prompt'],refs,20261113+i)
    result={"id":q['id'],"category":q['category'],"prompt":q['prompt'],
        "reference_ids_for_review":q['reference_ids'],
        "retrieved_ids":[r['id'] for r in refs],
        "retrieval_similarities":[r['similarity'] for r in refs],
        "no_context":base,"retrieved_context":grounded}
    results.append(result)
    OUT.write_text(json.dumps({"version":bank['version'],"model":MODEL_ID,"results":results,
        "research_only":True,"context_type":"automatic_tfidf_retrieval","production_approved":False},ensure_ascii=False,indent=2),encoding='utf-8')
    print(f"{len(results)}/{QUESTION_LIMIT} {q['id']} | bare: {base['answer'][:100]} | grounded: {grounded['answer'][:100]}",flush=True)
print("Paired comparisons saved:",len(results),"at",OUT)


## 5 — Report, retrieval diagnostics and human-review CSV

We calculate **retrieval hit rate** and operational diagnostics such as truncation/repetition, but **not automatic factual-quality scores**. Human reviewers must score both answers for factual accuracy, natural Hindi, and instruction following (0–2 each), considering possible hallucinations and whether retrieved passages are correct.

A higher retrieval hit rate is not itself proof that grounded answers are better. Your human-review CSV is left blank to avoid fabricating results.


In [ ]:
import csv, statistics
from google.colab import files
complete=len(results)==len(questions)
def summarize(key):
    vals=[r[key] for r in results]
    return {"answers":len(vals),
        "hit_limit":sum(bool(x['hit_limit']) for x in vals),
        "stopped_on_eos":sum(bool(x['stopped_on_eos']) for x in vals),
        "with_replacement_characters":sum(x['replacement_characters']>0 for x in vals),
        "high_repetition":sum(x['repetition_ratio']>0.22 for x in vals),
        "avg_output_tokens":round(statistics.mean(x['tokens'] for x in vals),2) if vals else None}
hits=sum(bool(set(r['retrieved_ids'])&set(r['reference_ids_for_review'])) for r in results)
report={"version":bank['version'],"model":MODEL_ID,"completed_pairs":len(results),
        "total_questions":len(questions),"complete":complete,
        "context_type":"automatically_retrieved_candidate_reference_text",
        "knowledge_status":knowledge['status'],
        "knowledge_sha256":hashlib.sha256((ROOT/'coreforge_eval_v2_knowledge.json').read_bytes()).hexdigest(),
        "question_bank_sha256":hashlib.sha256((ROOT/'coreforge_eval_v2_questions.json').read_bytes()).hexdigest(),
        "generation":{"max_new_tokens":160,"temperature":0.7,"top_p":0.8,"top_k":20},
        "retrieval":{"any_expected_reference_in_top2":hits,"rate":round(hits/len(results),3) if results else None},
        "no_context":summarize('no_context'),"retrieved_context":summarize('retrieved_context'),
        "production_approved":False,"human_quality_assessment":"pending","results":results}
FINAL=ROOT/'coreforge-eval-v2-results.json'
REVIEW=ROOT/'coreforge-eval-v2-human-review.csv'
FINAL.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
with REVIEW.open('w',encoding='utf-8',newline='') as f:
    w=csv.writer(f)
    w.writerow(["id","prompt","expected_references","retrieved_references","no_context_answer","grounded_answer",
      "no_context_accuracy_0_2","grounded_accuracy_0_2","no_context_hindi_0_2","grounded_hindi_0_2",
      "no_context_instruction_0_2","grounded_instruction_0_2","human_notes"])
    for r in results:
        w.writerow([r['id'],r['prompt'],','.join(r['reference_ids_for_review']),','.join(r['retrieved_ids']),
            r['no_context']['answer'],r['retrieved_context']['answer'],"","","","","","",""])
print("Complete:",complete,"| Retrieval hits:",hits,"/",len(results))
print("No context:",report['no_context'])
print("Retrieved context:",report['retrieved_context'])
files.download(str(FINAL))
files.download(str(REVIEW))


## What to send back
Upload **coreforge-eval-v2-results.json** here. You may also fill in and upload the human-review CSV. We will compare actual answer quality, look for unsupported or misleading reference passages, and decide whether the next step should be a document-based QA assistant, a revised reference corpus, or additional fine-tuning.

**Never deploy this candidate knowledge set as official Adatacore policy without domain-owner review.**
